# Polars Data Operations Demo

This notebook demonstrates how to use Polars for common data operations on UK Biobank proteomics and phenotype data, equivalent to the PySpark operations in the original notebook.

In [21]:
import polars as pl
import numpy as np

# Configure Polars for better display
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_width_chars(200)

polars.config.Config

In [ ]:
# Read parquet files with Polars (equivalent to Spark spark.read.parquet)
# Use glob pattern to exclude .crc files created by Spark
olink_df = pl.read_csv("/mnt/disk2/dataset/ukb/olink0_data.csv")
hp_df = pl.read_csv("/mnt/disk2/dataset/ukb/hypertension.csv")



print(f"Olink shape: {olink_df.shape}")
print(f"Hypertension shape: {hp_df.shape}")

Olink shape: (52995, 2924)
Hypertension shape: (22987, 10)


In [34]:
hp_df.write_parquet('/mnt/disk2/dataset/ukb/hypertension.parquet')

## Use Apache Iceberg for Data Management

In [29]:
# Write in Iceberg
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, StringType, DoubleType

catalog = load_catalog(
    "rest",
    **{
        "type": "rest",
        "uri": "http://localhost:8181",
        "s3.endpoint": "http://localhost:9900",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
    },
)


In [27]:
catalog.list_namespaces()

[('ukb',)]

In [30]:
# catalog.create_namespace('ukb')
schema = olink_df.collect_schema().to_arrow()
olink_instance_0_table = catalog.create_table_if_not_exists(
    identifier="ukb.olink_instance_0",
    schema=schema
)
olink_df.write_iceberg(olink_instance_0_table, mode="overwrite")

In [33]:
catalog.list_tables('ukb')

[('ukb', 'olink_instance_0')]

In [ ]:
# Explore olink data (equivalent to olink_df.limit(10).toPandas())
olink_df.head(10)

eid,a1bg,aamdc,aarsd1,abca2,abhd14b,abl1,abo,abraxas2,acaa1,acadm,acadsb,acan,ace,ace2,ache,acot13,acox1,acp1,acp5,acp6,acrbp,acrv1,acsl1,acta2,actn2,actn4,acvrl1,acy1,acy3,acyp1,ada,ada2,adam12,adam15,adam22,adam23,…,wasl,wdr46,wfdc1,wfdc12,wfdc2,wfikkn1,wfikkn2,wif1,wnt9a,wwp2,xcl1,xg,xiap,xpnpep2,xrcc4,yap1,yars1,yes1,yju2,yod1,ythdf3,ywhaq,yy1,zbp1,zbtb16,zbtb17,zcchc8,zfyve19,zhx2,znf174,znf75d,znf830,znrd2,znrf4,zp3,zp4,zpr1
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000190,null,null,0.30215,null,null,1.01985,null,null,0.2029,null,null,1.0714,null,0.6189,null,null,1.32895,null,0.74765,0.80215,null,null,null,0.6948,null,0.04235,null,1.09045,null,null,0.03685,-0.16785,null,-0.1713,null,0.35365,…,null,null,null,-0.002,-0.0468,null,0.29605,-0.1376,0.1351,null,-0.4858,0.35065,null,0.14545,0.19865,null,null,1.2325,null,null,1.33885,null,null,null,0.933,0.34205,null,null,null,null,null,null,null,null,null,null,null
1000561,null,null,-0.30405,null,-1.70415,-0.98305,null,null,-0.6176,null,null,0.5892,null,-0.0275,null,null,-0.29895,null,-0.51075,-0.26355,null,null,null,0.239,null,-0.19315,0.1447,-0.75035,null,null,-0.01035,-0.08365,null,0.3496,0.605,0.25215,…,null,null,null,0.7442,0.1054,0.138,0.63895,0.4637,0.2416,-1.4509,-0.1622,0.51705,null,0.35555,-0.39605,null,null,-2.5894,null,null,-2.43985,null,null,null,-1.1629,-0.66235,null,null,null,null,null,null,null,null,null,null,null
1000751,0.0915,-0.54025,-0.22905,0.0727,-0.20995,0.0082,-0.2633,0.2647,-0.11335,-0.0662,0.10195,-0.0527,0.01435,null,0.0013,-0.0037,-0.065,-0.1635,-0.8152,-0.1125,-0.0066,-0.25765,0.3064,null,0.2066,2.87995,0.1181,0.31295,0.4967,-0.5333,-0.315,0.3312,0.6322,null,0.1418,0.5422,…,-0.7692,0.1621,0.4793,0.1092,1.2601,-0.1666,0.2539,0.38725,0.586,-0.0735,0.1188,0.5338,0.0192,0.352,0.0472,0.6145,0.0313,0.5437,0.2001,-0.3961,0.4204,0.0718,0.4433,0.03875,-0.04725,null,-0.3238,0.3745,0.0456,0.216,-0.2987,0.3727,0.19135,-0.0679,0.3216,0.4027,-0.598
1001566,-0.0513,-0.25025,-0.0291,0.19195,-0.5609,-0.0932,0.5822,0.3781,-0.9033,-0.0654,-0.23465,0.1053,0.23085,-0.49075,0.0392,-0.4907,-0.2092,-0.19325,0.1244,0.23,0.1232,-0.45565,0.8866,-0.7563,-0.1294,0.04695,-0.2827,-0.4833,0.3042,-0.3743,-0.1643,0.1629,0.3288,0.16805,0.0146,-0.0082,…,0.1108,0.1205,0.0431,0.4091,0.2216,0.11565,0.3883,0.08145,0.6543,-0.0857,0.2303,0.5478,-0.4298,-1.3687,0.34745,-0.3104,-0.5814,-0.1396,0.0342,-0.2648,-0.34235,-0.3083,0.0485,-0.48155,-0.04605,-0.3924,0.3346,-0.0858,0.0801,-0.1634,-0.1477,1.9119,-0.22735,0.1714,-4.3227,0.1498,0.6211
1002213,0.1618,0.12725,0.00005,-0.3617,0.05145,0.3401,-1.7924,-0.3592,0.5233,-0.3454,0.54525,-0.26905,-0.14175,0.2505,-0.3733,-0.5635,-0.5388,-0.3563,0.2654,-0.4031,0.1682,-0.54585,-0.0932,-0.10195,-0.5712,0.29325,-0.1999,-0.4178,-0.1643,0.2667,0.1374,0.6089,-0.0657,-0.24945,-0.48655,-0.2199,…,-0.2502,0.2585,-0.0612,0.0871,0.2333,-0.9185,-0.3135,0.29085,0.569,0.0898,-0.76745,0.4481,-0.1324,-0.1914,0.22535,0.0262,0.5819,0.3271,-0.1952,0.0595,0.4001,0.0913,-0.7093,-0.35565,0.04115,-0.3758,-0.6546,0.3028,-0.1166,-0.0475,0.0888,-0.224,-0.60155,-0.1991,-4.1014,-0.143,-0.6588
1002842,0.0267,-0.19305,-0.00095,0.0672,0.6114,0.6274,1.7538,-0.937,-0.1719,-0.0305,-1.16535,-0.0336,0.61695,0.72295,0.3325,-0.9749,0.0028,-0.078,0.6221,0.7812,0.0106,4.12775,0.083,0.1116,0.6204,0.042,0.0977,-0.3561,-0.5222,-0.5856,0.5913,0.098,-0.0099,0.33775,0.123,0.35595,…,0.4675,-0.0033,-0.1453,-0.4238,0.269,0.394,0.1388,0.16175,-0.2563,0.5272,0.733,-0.0522,0.1908,-0.3097,0.35985,-0.1715,-0.9623,0.4441,0.0,-0.1647,-0.27495,-0.2533,0.1561,0.01495,-0.00145,-0.0031,0.3737,-0.1596,0.1182,0.005,0.524,-0.0596,0.30505,-0.1135,-1.0352,0.0,-0.2845
1002982,-0.0006,-0.94585,-0.28155,-1.5006,-0.4544,-0.5623,1.4414,-0.1872,-1.2167,-0.1714,-0.63775,-

In [11]:
# Explore hypertension data
hp_df.head(10)

,participant.eid,participant.p131286,participant.p21003_i0,participant.p21001_i0,participant.p48_i0,participant.p1239_i0,participant.p1249_i0,participant.p2976_i0,olink_instance_0.eid
i64,i64,str,i64,f64,f64,f64,f64,f64,i64
0,1003370,"""2013-07-01""",66,29.957936,109.0,0.0,1.0,null,1003370
1,1003757,"""2014-08-14""",46,45.105601,108.0,0.0,4.0,null,1003757
2,1004168,"""2013-04-01""",56,26.367087,93.0,0.0,1.0,null,1004168
3,1004587,"""2001-09-01""",59,33.184348,101.0,0.0,1.0,null,1004587
4,1007786,"""2008-09-01""",59,43.510937,117.0,0.0,4.0,null,1007786
5,1007995,"""1998-08-12""",55,25.238535,87.0,0.0,3.0,null,1007995
6,1011323,"""2000-11-01""",62,36.884288,106.0,0.0,1.0,null,1011323
7,1014157,"""2007-11-01""",66,30.59086,101.0,0.0,4.0,null,1014157
8,1016393,"""1994-05-01""",65,24.454086,87.0,0.0,3.0,null,1016393


In [10]:
# View schema (equivalent to hp_df.printSchema())
print("Olink schema:")
print(olink_df.schema)
print("\nHypertension schema:")
print(hp_df.schema)

Olink schema:
Schema({'eid': Int64, 'a1bg': Float64, 'aamdc': Float64, 'aarsd1': Float64, 'abca2': Float64, 'abhd14b': Float64, 'abl1': Float64, 'abo': Float64, 'abraxas2': Float64, 'acaa1': Float64, 'acadm': Float64, 'acadsb': Float64, 'acan': Float64, 'ace': Float64, 'ace2': Float64, 'ache': Float64, 'acot13': Float64, 'acox1': Float64, 'acp1': Float64, 'acp5': Float64, 'acp6': Float64, 'acrbp': Float64, 'acrv1': Float64, 'acsl1': Float64, 'acta2': Float64, 'actn2': Float64, 'actn4': Float64, 'acvrl1': Float64, 'acy1': Float64, 'acy3': Float64, 'acyp1': Float64, 'ada': Float64, 'ada2': Float64, 'adam12': Float64, 'adam15': Float64, 'adam22': Float64, 'adam23': Float64, 'adam8': Float64, 'adam9': Float64, 'adamts1': Float64, 'adamts13': Float64, 'adamts15': Float64, 'adamts16': Float64, 'adamts4': Float64, 'adamts8': Float64, 'adamtsl2': Float64, 'adamtsl4': Float64, 'adamtsl5': Float64, 'adcyap1r1': Float64, 'add1': Float64, 'adgrb3': Float64, 'adgrd1': Float64, 'adgre1': Float64, 'a

In [11]:
# Descriptive statistics (equivalent to hp_df.describe().show())
# Note: Polars describe returns a DataFrame, not printed as table
hp_df.describe()

statistic,,participant.eid,participant.p131286,participant.p21003_i0,participant.p21001_i0,participant.p48_i0,participant.p1239_i0,participant.p1249_i0,participant.p2976_i0,olink_instance_0.eid
str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64
"""count""",22987.0,22987.0,"""22987""",22987.0,22838.0,22905.0,22960.0,21123.0,2424.0,22987.0
"""null_count""",0.0,0.0,"""0""",0.0,149.0,82.0,27.0,1864.0,20563.0,0.0
"""mean""",11493.0,3.5133e6,null,59.604385,28.956066,95.116333,0.127918,2.601146,49.023515,3.5133e6
"""std""",6635.919655,1.4603e6,null,7.302253,5.089142,13.589634,0.412219,1.351679,16.034832,1.4603e6
"""min""",0.0,1.000034e6,"""1902-02-02""",40.0,15.851242,57.0,-3.0,-3.0,-3.0,1.000034e6
"""25%""",5747.0,2.23094e6,null,55.0,25.469388,86.0,0.0,1.0,45.0,2.23094e6
"""50%""",11493.0,3.524509e6,null,61.0,28.200074,95.0,0.0,3.0,54.0,3.524509e6
"""75%""",17240.0,4.790324e6,null,65.0,31.584532,104.0,0.0,4.0,60.0,4.790324e6
"""max""",22986.0,6.021398e6,"""2025-07-01""",70.0,68.950501,173.0,2.0,4.0,69.0,6.021398e6


In [ ]:
# Rename columns: replace '.' with '_' (equivalent to PySpark reduce logic)
hp_df = hp_df.rename({col: col.replace(".", "_") for col in hp_df.columns})
hp_df.columns

In [ ]:
# Select specific columns
hp_df.select(["participant_eid", "participant_p21001_i0", "participant_p48_i0"]).head()

In [ ]:
# Filter rows (equivalent to SQL WHERE)
# Find participants with BMI > 30
hp_df.filter(pl.col("participant_p21001_i0") > 30).head()

In [ ]:
# Filter rows with missing values in specific column
hp_df.filter(pl.col("participant_p21001_i0").is_null()).height

In [ ]:
# Join datasets (equivalent to Spark join)
# First, standardize column names for join
olink_renamed = olink_df.rename({"eid": "participant_eid"})

# Inner join on participant_eid
joined_df = olink_renamed.join(hp_df, on="participant_eid", how="inner")
print(f"Joined shape (inner): {joined_df.shape}")
joined_df.head(3)

In [ ]:
# Left join - keep all olink participants
left_joined = olink_renamed.join(hp_df, on="participant_eid", how="left")
print(f"Left joined shape: {left_joined.shape}")

In [ ]:
# Add new column (equivalent to withColumn)
# Create BMI category
hp_with_bmi_cat = hp_df.with_columns(
    pl.when(pl.col("participant_p21001_i0") < 18.5).then(pl.lit("underweight"))
    .when(pl.col("participant_p21001_i0") < 25).then(pl.lit("normal"))
    .when(pl.col("participant_p21001_i0") < 30).then(pl.lit("overweight"))
    .otherwise(pl.lit("obese"))
    .alias("bmi_category")
)
hp_with_bmi_cat.select(["participant_eid", "participant_p21001_i0", "bmi_category"]).head()

In [ ]:
# Group by and aggregate (equivalent to Spark groupBy)
# Count by BMI category
hp_with_bmi_cat.group_by("bmi_category").agg(
    pl.count().alias("count"),
    pl.col("participant_p21001_i0").mean().alias("mean_bmi")
).sort("count", descending=True)

In [ ]:
# Handle missing values
# Fill null with mean for a protein column
protein_col = "a1bg"
mean_val = olink_df[protein_col].mean()
olink_filled = olink_df.with_columns(
    pl.col(protein_col).fill_null(mean_val)
)
print(f"Nulls in {protein_col} before: {olink_df[protein_col].null_count()}")
print(f"Nulls in {protein_col} after: {olink_filled[protein_col].null_count()}")

In [ ]:
# Reshape data: melt from wide to long format (useful for GWAS)
# Select a few proteins for demo
protein_cols = ["a1bg", "aamdc", "aarsd1", "abca2"]
olink_melted = olink_df.select(["eid"] + protein_cols).melt(
    id_vars=["eid"],
    variable_name="protein",
    value_name="expression"
)
print(f"Melted shape: {olink_melted.shape}")
olink_melted.head()

In [ ]:
# Pivot back to wide format
olink_pivoted = olink_melted.pivot(
    on="protein",
    index="eid",
    values="expression"
)
print(f"Pivoted shape: {olink_pivoted.shape}")
olink_pivoted.head()

In [ ]:
# Write to parquet (parallel, fast)
# hp_df.write_parquet("output/hypertension_processed.parquet")

# Or write to CSV
# hp_df.write_csv("output/hypertension_processed.csv")

In [ ]:
# Use LazyFrame for query optimization (equivalent to Spark's lazy evaluation)
lazy_hp = hp_df.lazy()

result = (
    lazy_hp
    .filter(pl.col("participant_p21001_i0") > 30)
    .group_by("participant_p1249_i0")
    .agg(
        pl.count().alias("n"),
        pl.col("participant_p21001_i0").mean().alias("mean_bmi")
    )
    .sort("n", descending=True)
)

# Collect to execute
result.collect()

## Summary: Polars vs PySpark Equivalents

| Operation | PySpark | Polars |
|----------|---------|--------|
| Read data | `spark.read.parquet()` | `pl.read_parquet()` |
| View schema | `.printSchema()` | `.schema` |
| Describe | `.describe().show()` | `.describe()` |
| Select cols | `.select()` | `.select()` |
| Rename cols | `.withColumnRenamed()` | `.rename()` |
| Filter | `.filter()` | `.filter()` |
| Add column | `.withColumn()` | `.with_columns()` |
| Join | `.join()` | `.join()` |
| Group by | `.groupBy()` | `.group_by()` |
| Fill null | `.fillna()` | `.fill_null()` |
| Write | `.write.parquet()` | `.write_parquet()` |
| Lazy evaluation | DataFrame (lazy) | `lazy()` / `collect()` |